In [18]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.linear = nn.Linear(10, 10)

    def forward(self, x):
        return self.linear(x)

model = MyModel()
example_input = torch.randn(1, 10)
# traced_model = torch.jit.trace(model, example_input)
scripted_model = torch.jit.script(model)


optimizer = torch.optim.SGD(scripted_model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# Exemple d'entraînement
for epoch in range(100):
    optimizer.zero_grad()
    output = scripted_model(example_input)
    loss = loss_fn(output, torch.randn(1, 10))
    loss.backward()
    optimizer.step()

In [16]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import torch

def amax(x, block_size):
        block_shape = list(x.shape[:-1]) + [x.shape[-1] // block_size] + [block_size]
        block_tensor = x.view(block_shape)
        amax = torch.amax(torch.abs(block_tensor), dim=-1, keepdim=True)
        # amax.repeat_interleave(block_size, dim=-1)
        return amax

x = torch.Tensor([
    [[1, 2, 3, 4],
     [5, 6, 7, 8],
     [9, 10, 11, 12]],

    [[13, 14, 15, 16],
     [17, 18, 19, 20],
     [21, 22, 23, 24]]
])

block_size = 2

abs_max = amax(x, block_size)

block_shape = list(x.shape[:-1]) + [x.shape[-1] // block_size] + [block_size]
block_tensor = x.view(block_shape)
r = (block_tensor/abs_max).view(x.shape)
print(r)

tensor([[[0.5000, 1.0000, 0.7500, 1.0000],
         [0.8333, 1.0000, 0.8750, 1.0000],
         [0.9000, 1.0000, 0.9167, 1.0000]],

        [[0.9286, 1.0000, 0.9375, 1.0000],
         [0.9444, 1.0000, 0.9500, 1.0000],
         [0.9545, 1.0000, 0.9583, 1.0000]]])


In [10]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    "deepseek-ai/deepseek-coder-1.3b-base", trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-1.3b-base", trust_remote_code=True
)

# Définir la couche personnalisée
class CustomLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(CustomLinear, self).__init__()
        # self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        # Renvoyer toujours zéro
        return torch.zeros_like(x)

# Fonction pour remplacer les couches linéaires par des couches personnalisées
# def replace_linear_layers(model):
#     for name, module in model.named_children():
#         if isinstance(module, nn.Linear):
#             # Remplacer la couche linéaire par la couche personnalisée
#             setattr(model, name, CustomLinear(module.in_features, module.out_features))
#         elif isinstance(module, nn.Module):
#             # Si le module est un sous-module, appliquer la fonction récursivement
#             replace_linear_layers(module)

def replace_linear_layers(
    model: nn.Module,
    layers_quant_type: dict | str = "01",
):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            if isinstance(layers_quant_type, str):
                quantized_layer = CustomLinear(
                    in_features=module.in_features,
                    out_features=module.out_features,
                )
            else:
                for layer, quantize_mode in layers_quant_type.items():
                    if name.endswith(layer):
                        quantized_layer = CustomLinear(
                            in_features=module.in_features,
                            out_features=module.out_features,
                        )
            setattr(model, name, quantized_layer)
        elif isinstance(module, nn.Module):
            replace_linear_layers(module, layers_quant_type)


# Exemple de modèle avec des sous-modules
class SubModule(nn.Module):
    def __init__(self):
        super(SubModule, self).__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 30)

    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        return x

class ComplexModel(nn.Module):
    def __init__(self):
        super(ComplexModel, self).__init__()
        self.submodule1 = SubModule()
        self.submodule2 = SubModule()
        self.fc3 = nn.Linear(30, 40)

    def forward(self, x):
        x1 = self.submodule1(x)
        x2 = self.submodule2(x)
        x = x1 + x2
        x = self.fc3(x)
        return x

# Créer une instance du modèle complexe
# model = ComplexModel()

# Remplacer les couches linéaires par des couches personnalisées
replace_linear_layers(model)

# Vérifier que les couches ont été remplacées
print(model)

# Tester le modèle avec une entrée
input_tensor = torch.randn(1, 10)
output = model(input_tensor)
print(output)  # Devrait afficher un tenseur de zéros

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32256, 2048)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): CustomLinear()
          (k_proj): CustomLinear()
          (v_proj): CustomLinear()
          (o_proj): CustomLinear()
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): CustomLinear()
          (up_proj): CustomLinear()
          (down_proj): CustomLinear()
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-06)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_head): CustomLinear()
)


RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

In [5]:
import torch
import time

# Exemple de tenseurs
batch_size = 200000
len = 30
dim = 40

x = torch.randn(batch_size, len, dim)
y = torch.randint(0, 2, (dim, dim), dtype=torch.bool)
# y = torch.randn(dim, dim)

# Mesure du temps pour la multiplication normale
start_time = time.time()
masked_x_normal = x @ y.to(torch.float32)
normal_time = time.time() - start_time

# Affichage du temps d'exécution
print(f"Temps d'exécution pour la multiplication normale : {normal_time:.6f} secondes")

# # Mesure du temps pour la multiplication optimisée
# start_time = time.time()
# masked_x_optimized = y * x.unsqueeze(-1)
# result_optimized = masked_x_optimized.sum(dim=-2)
# optimized_time = time.time() - start_time

# # Affichage du temps d'exécution
# print(f"Temps d'exécution pour la multiplication optimisée : {optimized_time:.6f} secondes")

# # Vérification que les résultats sont identiques (optionnel)
# print(f"Les résultats sont-ils identiques ? {torch.allclose(masked_x_normal, result_optimized)}")

Temps d'exécution pour la multiplication normale : 0.793141 secondes


In [2]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

c:\Users\Chris\Documents\SuperQuantization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [ ]:
print(len(ds['validation']['text']))

21990


: 

In [3]:
import random

# Fonction pour tester les inégalités
def test_inequalities(num_tests=10000):
    all_satisfied = True
    failed_cases = []
    
    for _ in range(num_tests):
        # Générer 6 nombres aléatoires
        A = random.uniform(-100, 100)
        B = random.uniform(-100, 100)
        C = random.uniform(-100, 100)
        a = random.uniform(-100, 100)
        b = random.uniform(-100, 100)
        c = random.uniform(-100, 100)
        
        # Vérifier les deux premières inégalités
        if (A - b)**2 > (a - B)**2 and (B - c)**2 > (c - C)**2:
            # Tester la troisième inégalité
            if not (C - a)**2 < (c - A)**2:
                all_satisfied = False
                failed_cases.append((A, B, C, a, b, c))
        else:
            continue

    return all_satisfied, failed_cases

# Effectuer les tests
num_tests = 10000
result, failures = test_inequalities(num_tests)

# Afficher les résultats
if result:
    print(f"Toutes les inégalités sont respectées pour {num_tests} tests.")
else:
    print(f"L'inégalité (C-a)^2 < (c-A)^2 a échoué dans {len(failures)} cas.")
    print("Exemples de cas ayant échoué :", failures[:5])  # Afficher quelques exemples d'échec
import random

# Fonction pour tester les inégalités
def test_inequalities(num_tests=10000):
    all_satisfied = True
    failed_cases = []
    
    for _ in range(num_tests):
        # Générer 6 nombres aléatoires
        A = random.uniform(-100, 100)
        B = random.uniform(-100, 100)
        C = random.uniform(-100, 100)
        a = random.uniform(-100, 100)
        b = random.uniform(-100, 100)
        c = random.uniform(-100, 100)
        
        # Vérifier les deux premières inégalités
        if (A - b)**2 > (a - B)**2 and (B - c)**2 > (b - C)**2:
            # Tester la troisième inégalité
            if not (C - a)**2 < (c - A)**2:
                all_satisfied = False
                failed_cases.append((A, B, C, a, b, c))
        else:
            continue

    return all_satisfied, failed_cases

# Effectuer les tests
num_tests = 10000
result, failures = test_inequalities(num_tests)

# Afficher les résultats
if result:
    print(f"Toutes les inégalités sont respectées pour {num_tests} tests.")
else:
    print(f"L'inégalité (C-a)^2 < (c-A)^2 a échoué dans {len(failures)} cas.")
    print("Exemples de cas ayant échoué :", failures[:5])  # Afficher quelques exemples d'échec


L'inégalité (C-a)^2 < (c-A)^2 a échoué dans 1162 cas.
Exemples de cas ayant échoué : [(97.0086397709727, -34.18441283735159, 87.83950119013596, 11.954577832132827, 29.594910520236624, 35.72267868749316), (-14.929859707128102, 86.4920848604086, 26.37808895453584, 82.93970272497106, 92.70318381863984, -53.531035403357066), (-45.79159095080036, 30.296029715845037, -80.42912351864203, -11.25101049073156, 67.91211859881997, -85.65032821498104), (26.076862854321647, 53.45033492213852, -67.51170594563978, 73.87153237452091, -82.57667190008434, -13.088697489441998), (-0.8311323439084788, 73.61750310962623, -5.210757750440024, 87.01955799102936, 64.06249482576172, -41.38448343863574)]
L'inégalité (C-a)^2 < (c-A)^2 a échoué dans 1084 cas.
Exemples de cas ayant échoué : [(36.959399780567054, -20.24367677471936, -14.885539651136085, 31.112061651562414, -69.66551037560158, 69.68660180663733), (-59.478823053842376, 68.68739590535452, -93.38018455237082, 61.321722817864895, -46.650055191258375, -91.2

In [5]:
import json
import os

with open('../data/squad/train-v2.0.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

In [51]:
print(data['data'][0]['paragraphs'][55]['qas'][10]['question'])

# On va faire 'context'

Who asked her to change her mind about the soft drink deal due to the nature of the product?


In [61]:
print(data['data'][10]['paragraphs'][55]['qas'][1]['answers'])

[{'text': 'Obama', 'answer_start': 682}]
